# Notebook 8: Principal Component Analysis (PCA)

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 75 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain PCA intuitively using the photography analogy
2. Derive PCA via SVD in plain English — and implement it from scratch
3. Read and interpret a scree plot to choose the right number of components
4. Apply PCA for 2D/3D visualisation of high-dimensional data
5. Use PCA for noise reduction and reconstruct images at varying compression levels
6. Build a biplot to interpret which original features drive each principal component

---

**Week 11 Theme:** Customer segmentation / retail data

## The Photography Analogy

Imagine you are photographing a **3D sculpture** in a gallery. Your camera produces a flat, 2D image — a projection of the 3D object onto a plane. You want to choose the angle that captures the most detail.

- If you photograph the sculpture **from the front**, you capture its widest, most informative view.
- If you then take a second photo **from the side** (perpendicular to the first), you capture the next most informative view.
- A photo from directly above captures the least additional information because most of the sculpture's variation is already shown in the first two photos.

**PCA does exactly this for data:**

- The **first principal component (PC1)** is the direction in the original high-dimensional feature space that captures the most **variance** in the data — the "widest view".
- **PC2** is the next-best direction that is *perpendicular* (orthogonal) to PC1.
- **PC3** is perpendicular to both PC1 and PC2, and captures the third-most variance.
- And so on.

By keeping only the first *k* principal components, you create the best possible *k*-dimensional "photo" of your data, discarding only the least informative detail.

## Why Does This Matter for ML?

High-dimensional data is everywhere in ML — images, text embeddings, sensor readings, customer feature matrices. PCA addresses several problems simultaneously:

| Problem | How PCA Helps |
|---------|---------------|
| **Visualisation** | Project to 2D or 3D so you can plot and explore cluster structure |
| **Noise reduction** | Signal lives in the top PCs; projecting back and forth filters noise in the remaining PCs |
| **Feature compression** | Replace 64 correlated pixel values with 10 PCs that capture 95% of the variance |
| **Multicollinearity** | PCs are uncorrelated by construction — safe to feed into models that assume independence |
| **Speed** | Fewer features → faster training for downstream models |

**Business context:** A retail dataset with 50 customer features is hard to segment or visualise. PCA lets you project the data to 2 dimensions to spot natural customer groupings before you ever run K-Means.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import FancyArrowPatch

np.random.seed(42)                       # reproducibility
sns.set_theme(style='whitegrid')         # consistent plot style

print('Libraries loaded successfully.')

## The Variance-Maximisation View

### What PCA is actually doing

Suppose your data matrix $X$ has $n$ rows (data points) and $d$ columns (features). You want to project each point onto a 1D line through the origin and maximise how spread-out (how much *variance*) the projected points have.

It turns out the best direction to project onto is the **eigenvector of the covariance matrix** corresponding to the largest eigenvalue. The second-best direction is the eigenvector with the second-largest eigenvalue, and so on.

### Three equivalent ways to derive PCA

1. **Maximise variance of projections** — find the direction $\mathbf{w}$ such that $\text{Var}(X\mathbf{w})$ is largest. Solution: leading eigenvector of $X^\top X$.

2. **Minimise reconstruction error** — find the low-dimensional representation that lets you reconstruct the original data with the smallest mean squared error. Same solution.

3. **SVD of the centred data matrix** — compute $X_c = X - \bar{X}$ (subtract the mean), then factorise $X_c = U \Sigma V^\top$. The columns of $V$ are the principal components (the directions). This is what we implement in code.

All three derivations give the same answer. The SVD approach is the most numerically stable and is what sklearn uses internally.

In [ ]:
np.random.seed(42)

# Generate 2D correlated data to visualise the PCA axes
mean = [0, 0]
cov = [[3.0, 2.5],    # strong positive correlation between the two features
       [2.5, 2.5]]

X_2d = np.random.multivariate_normal(mean, cov, size=300)

# Compute covariance matrix and its eigenvectors
X_centered = X_2d - X_2d.mean(axis=0)
cov_matrix = np.cov(X_centered.T)             # (2,2) covariance matrix
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)  # sorted ascending

# Reverse to descending order (largest eigenvalue first)
eigenvalues = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]          # columns are eigenvectors

pc1 = eigenvectors[:, 0]   # first principal component direction
pc2 = eigenvectors[:, 1]   # second principal component direction

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.4, s=20, color='steelblue', label='Data')

# Draw PC1 and PC2 as arrows scaled by sqrt(eigenvalue) to show the spread
scale1 = np.sqrt(eigenvalues[0]) * 2
scale2 = np.sqrt(eigenvalues[1]) * 2

ax.annotate('', xy=pc1 * scale1, xytext=-pc1 * scale1,
            arrowprops=dict(arrowstyle='<->', color='crimson', lw=2.5))
ax.annotate('', xy=pc2 * scale2, xytext=-pc2 * scale2,
            arrowprops=dict(arrowstyle='<->', color='darkorange', lw=2.5))

ax.text(*(pc1 * scale1 * 1.1), 'PC1\n(max variance)', color='crimson', fontsize=11,
        ha='center', fontweight='bold')
ax.text(*(pc2 * scale2 * 1.3), 'PC2\n(⊥ to PC1)', color='darkorange', fontsize=11,
        ha='center', fontweight='bold')

ax.set_aspect('equal')
ax.set_title('2D Intuition: PCA finds the axes of the data ellipse')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.show()

print(f'Eigenvalue 1 (variance along PC1): {eigenvalues[0]:.3f}')
print(f'Eigenvalue 2 (variance along PC2): {eigenvalues[1]:.3f}')
print(f'PC1 explains {eigenvalues[0]/eigenvalues.sum()*100:.1f}% of total variance')

## Three Equivalent Derivations — Plain English Summary

### a. Variance Maximisation

"Find the unit vector $\mathbf{w}$ such that the variance of the projected data $X\mathbf{w}$ is maximised." Lagrange multipliers show the optimal $\mathbf{w}$ is the eigenvector of the covariance matrix $X^\top X / (n-1)$ with the largest eigenvalue.

### b. Reconstruction Error Minimisation

"Find the *k*-dimensional subspace such that projecting points onto it and back to the original space gives the smallest mean squared reconstruction error." This is the same subspace as in (a) — the span of the top *k* eigenvectors.

### c. Singular Value Decomposition (SVD) — what we implement

Centre the data: $X_c = X - \bar{X}$

Factorise: $X_c = U \Sigma V^\top$

- $U$ is $n \times d$: the left singular vectors (not directly used for PCA)
- $\Sigma$ is $d \times d$ diagonal: singular values $\sigma_1 \geq \sigma_2 \geq \ldots$
- $V^\top$ is $d \times d$: the **right singular vectors — the principal component directions**

The explained variance of PC *k* is $\sigma_k^2 / (n-1)$. To project data: $Z = X_c V_k$ where $V_k$ contains the first *k* rows of $V^\top$.

In [ ]:
np.random.seed(42)

class PCAScratch:
    """PCA implemented from scratch using numpy's SVD.
    API mirrors sklearn.decomposition.PCA for easy comparison.
    """

    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        self.mean_ = X.mean(axis=0)          # per-feature mean; shape (d,)
        X_centered = X - self.mean_          # centre the data

        # SVD: X_centered = U @ diag(S) @ Vt
        # full_matrices=False gives the economy (thin) SVD — faster, same result
        U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

        # The principal component directions are the ROWS of Vt (= columns of V)
        # Keep only the top n_components
        self.components_ = Vt[:self.n_components]   # shape: (n_components, d)

        # Explained variance: each singular value squared divided by (n-1)
        n = X.shape[0]
        self.explained_variance_ = (S ** 2) / (n - 1)   # shape: (min(n,d),)

        # Explained variance ratio: how much of the total variance each PC captures
        total_var = self.explained_variance_.sum()
        self.explained_variance_ratio_ = self.explained_variance_ / total_var

        return self

    def transform(self, X):
        """Project X onto the principal component subspace."""
        return (X - self.mean_) @ self.components_.T   # shape: (n, n_components)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, X_transformed):
        """Reconstruct data from the compressed representation."""
        # Multiply back by components and add the mean we subtracted
        return X_transformed @ self.components_ + self.mean_

print('PCAScratch class defined.')

In [ ]:
np.random.seed(42)

# Generate synthetic 3D data with strong structure in 2 dimensions
n = 200
t = np.linspace(0, 2 * np.pi, n)
X_3d = np.column_stack([
    3 * np.cos(t) + np.random.randn(n) * 0.3,   # feature 1: correlated with t
    2 * np.sin(t) + np.random.randn(n) * 0.3,   # feature 2: correlated with t
    np.random.randn(n) * 0.2                      # feature 3: pure noise
])

# Apply our scratch PCA
pca_s = PCAScratch(n_components=2)
X_proj_scratch = pca_s.fit_transform(X_3d)

# Apply sklearn PCA for comparison
pca_sk = PCA(n_components=2, random_state=42)
X_proj_sklearn = pca_sk.fit_transform(X_3d)

# Check explained variance ratios match (signs may differ — that is fine)
print('Explained variance ratio (scratch):', np.round(pca_s.explained_variance_ratio_[:2], 5))
print('Explained variance ratio (sklearn):', np.round(pca_sk.explained_variance_ratio_, 5))
print('Match:', np.allclose(pca_s.explained_variance_ratio_[:2],
                            pca_sk.explained_variance_ratio_, atol=1e-5))

# Components should match up to sign
print('\nComponents (scratch):')
print(np.round(pca_s.components_, 4))
print('Components (sklearn):')
print(np.round(pca_sk.components_, 4))

# Visualise projection
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_proj_scratch[:, 0], X_proj_scratch[:, 1],
                alpha=0.6, s=20, color='steelblue')
axes[0].set_title('PCA Projection (scratch)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_proj_sklearn[:, 0], X_proj_sklearn[:, 1],
                alpha=0.6, s=20, color='darkorange')
axes[1].set_title('PCA Projection (sklearn)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.suptitle('Scratch vs. sklearn: same subspace, signs may differ', fontsize=11)
plt.tight_layout()
plt.show()

## A Note on Sign Ambiguity

You may notice that the components produced by `PCAScratch` and sklearn have **opposite signs** for some components. For example, PC1 might be `[0.71, 0.70]` in one and `[-0.71, -0.70]` in the other.

This is completely normal and mathematically expected. If $\mathbf{v}$ is an eigenvector, then $-\mathbf{v}$ is also an eigenvector with the same eigenvalue. The **direction** of the subspace is the same — only the sign of the arrow is flipped.

**Practical consequence:** When comparing projections, the scatter plots may be **mirrored** left-right or top-bottom. The shape and relative distances are identical. sklearn uses a deterministic sign convention (`svd_flip`) to ensure reproducibility, but this is a convention, not a mathematical requirement.

In [ ]:
np.random.seed(42)

# Load the digits dataset: 1797 images, each 8x8 pixels = 64 features
digits = load_digits()
X_digits = digits.data           # shape: (1797, 64)
y_digits = digits.target         # shape: (1797,) — digit labels 0-9

# Fit PCA keeping all components so we can see the full scree plot
pca_full = PCA(random_state=42)
pca_full.fit(X_digits)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)   # cumulative variance
n_components_95 = np.searchsorted(cumvar, 0.95) + 1      # number of PCs for 95% variance

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: individual explained variance per component (scree plot)
axes[0].bar(range(1, 65), pca_full.explained_variance_ratio_,
            color='steelblue', alpha=0.7)
axes[0].set_title('Scree Plot: Variance per Component')
axes[0].set_xlabel('Principal Component index')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].axvline(x=n_components_95, color='crimson', linestyle='--',
                label=f'{n_components_95} PCs = 95% var')
axes[0].legend()

# Right: cumulative explained variance
axes[1].plot(range(1, 65), cumvar, color='steelblue', linewidth=2)
axes[1].axhline(y=0.95, color='crimson', linestyle='--', label='95% threshold')
axes[1].axvline(x=n_components_95, color='darkorange', linestyle=':',
                label=f'k = {n_components_95}')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of components k')
axes[1].set_ylabel('Cumulative Explained Variance Ratio')
axes[1].legend()

plt.suptitle('Digits Dataset (64 features): Scree Analysis', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nDigits dataset: {X_digits.shape[1]} original features')
print(f'Components needed to retain 95% variance: {n_components_95}')
print(f'Compression ratio: {n_components_95}/{X_digits.shape[1]} = {n_components_95/X_digits.shape[1]:.2%}')

In [ ]:
np.random.seed(42)

# Project digits to 2D for visualisation
pca_2d = PCA(n_components=2, random_state=42)
X_digits_2d = pca_2d.fit_transform(X_digits)

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(
    X_digits_2d[:, 0], X_digits_2d[:, 1],
    c=y_digits, cmap='tab10',
    s=15, alpha=0.7
)

# Add digit labels at cluster centres for readability
for digit in range(10):
    mask = y_digits == digit
    cx, cy = X_digits_2d[mask, 0].mean(), X_digits_2d[mask, 1].mean()
    ax.text(cx, cy, str(digit), fontsize=14, fontweight='bold',
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.colorbar(scatter, ax=ax, label='Digit class', ticks=range(10))
ax.set_title('Digits Dataset: First 2 Principal Components\n'
             f'(PC1 + PC2 capture {pca_2d.explained_variance_ratio_.sum()*100:.1f}% of variance)',
             fontsize=12)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

print('Notice: even without using labels, PCA has created partially separable clusters.')
print('Digits 0 and 6 are far from the centre; 3, 5, 8 overlap (harder to separate).')

## Explained Variance Ratio — Where Does It Come From?

### Plain English Derivation

After we compute the SVD $X_c = U \Sigma V^\top$, each singular value $\sigma_k$ measures how much the data is "stretched" along the *k*-th principal component direction.

The variance of the projection onto PC *k* is:

$$\text{Var}(\text{PC}_k) = \frac{\sigma_k^2}{n - 1}$$

The total variance of the centred data equals the sum of all these:

$$\text{Total Variance} = \sum_{k=1}^{d} \frac{\sigma_k^2}{n - 1}$$

Therefore the **explained variance ratio** of PC *k* is:

$$r_k = \frac{\sigma_k^2}{\sum_{j=1}^{d} \sigma_j^2}$$

The $(n-1)$ cancels, so the ratio only depends on the singular values. This is exactly what `pca.explained_variance_ratio_[k]` reports.

In [ ]:
np.random.seed(42)

# Demonstrate image reconstruction at different numbers of components
# Take a single digit image
img_idx = 0
original_img = X_digits[img_idx]   # shape: (64,) — flat 8x8 image

# Fit PCA with enough components for the reconstruction experiment
pca_reconstruct = PCA(n_components=64, random_state=42)
X_digits_transformed = pca_reconstruct.fit_transform(X_digits)

k_values = [2, 5, 10, 20, 64]   # number of components to use for reconstruction

fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(14, 3))

# Show original
axes[0].imshow(original_img.reshape(8, 8), cmap='gray_r', vmin=0, vmax=16)
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

# Show reconstructions at each k
for ax, k in zip(axes[1:], k_values):
    # Keep only the first k component scores; zero out the rest
    scores_k = X_digits_transformed[img_idx].copy()
    scores_k[k:] = 0                                          # discard higher components

    # Reconstruct: multiply by V (components) and add back the mean
    reconstructed = (scores_k @ pca_reconstruct.components_) + pca_reconstruct.mean_
    mse = np.mean((reconstructed - original_img) ** 2)       # reconstruction error

    ax.imshow(reconstructed.reshape(8, 8), cmap='gray_r', vmin=0, vmax=16)
    var_explained = pca_reconstruct.explained_variance_ratio_[:k].sum() * 100
    ax.set_title(f'k={k}\n{var_explained:.0f}% var\nMSE={mse:.1f}', fontsize=8)
    ax.axis('off')

plt.suptitle('Image Reconstruction Quality vs. Number of PCA Components',
             fontsize=11)
plt.tight_layout()
plt.show()

## PCA for Noise Reduction

### The Key Insight

If your **signal** (meaningful structure) lives in the top *k* principal components, then the remaining *d − k* components mostly capture **noise** — small, random fluctuations.

By:
1. Projecting your data onto the top *k* PCs
2. Projecting back to the original *d*-dimensional space

you effectively average out the noise. The reconstruction is a smoothed, denoised version of the original.

**Analogy:** Imagine recording a piano piece in a noisy room. The piano notes are the high-variance signal. The background hiss is low-variance noise. If you filter out the low-energy frequencies, you hear the piano more clearly. PCA does the same — it discards the "low energy" directions.

In [ ]:
np.random.seed(42)

# Add Gaussian noise to digits images
noise_level = 4.0   # standard deviation of the noise (pixel values range 0-16)
X_noisy = X_digits + np.random.normal(0, noise_level, X_digits.shape)
X_noisy = np.clip(X_noisy, 0, 16)   # clip to valid pixel range

# Denoise using PCA: fit on noisy data, reconstruct with k=20 components
k_denoise = 20
pca_denoise = PCA(n_components=k_denoise, random_state=42)
X_denoised = pca_denoise.inverse_transform(pca_denoise.fit_transform(X_noisy))

# Calculate MSE
mse_noisy = np.mean((X_noisy - X_digits) ** 2)
mse_denoised = np.mean((X_denoised - X_digits) ** 2)

print(f'MSE (noisy vs. original)   : {mse_noisy:.3f}')
print(f'MSE (denoised vs. original): {mse_denoised:.3f}')
print(f'Noise reduction: {(1 - mse_denoised/mse_noisy)*100:.1f}%')

# Show a few examples: original, noisy, denoised
n_show = 6
indices = [0, 1, 2, 3, 4, 5]

fig, axes = plt.subplots(3, n_show, figsize=(12, 5))
row_labels = ['Original', 'Noisy', f'Denoised (k={k_denoise})']
row_data = [X_digits, X_noisy, X_denoised]

for row_idx, (row_label, row_d) in enumerate(zip(row_labels, row_data)):
    for col_idx, img_i in enumerate(indices):
        axes[row_idx, col_idx].imshow(
            row_d[img_i].reshape(8, 8), cmap='gray_r', vmin=0, vmax=16
        )
        axes[row_idx, col_idx].axis('off')
    axes[row_idx, 0].set_ylabel(row_label, fontsize=10, rotation=90,
                                 labelpad=30, va='center')

plt.suptitle('PCA Denoising: Project to top-k PCs, reconstruct back', fontsize=11)
plt.tight_layout()
plt.show()

## PCA Assumptions and Limitations

### What PCA assumes

| Assumption | What it means |
|------------|---------------|
| **Linearity** | The important structure in the data can be described by linear combinations of features |
| **Orthogonality** | The principal components are uncorrelated (perpendicular to each other) |
| **Variance = information** | Directions with high variance are meaningful; low-variance directions are noise |

### When these assumptions fail

- **Non-linear structure:** If your data lies on a curve or manifold (e.g., a Swiss roll, a sphere), PCA will smear the structure. Use **t-SNE** or **UMAP** for visualisation, or **kernel PCA** for non-linear compression.
- **Outliers:** PCA maximises variance — large outliers dominate the first few PCs and can give a misleading picture of the data's true structure. Consider robust PCA or outlier removal first.
- **Scale sensitivity:** Features with large numerical ranges dominate PCA. **Always standardise** (zero mean, unit variance) before applying PCA.

### Key limitation to remember

PCA is **unsupervised** — it does not know about your target variable. A direction that explains a lot of variance in *X* may explain very little variance in *y*. For supervised dimensionality reduction, see **Linear Discriminant Analysis (LDA)**.

In [ ]:
np.random.seed(42)

# -----------------------------------------------------------------------
# Synthetic retail customer dataset (same as Notebook 7)
# Features: annual_spend, visit_frequency, avg_basket_size, loyalty_score, returns_rate
# -----------------------------------------------------------------------
feature_names = ['annual_spend', 'visit_freq', 'avg_basket', 'loyalty', 'returns']
n_customers = 300

c0 = np.random.multivariate_normal(
    mean=[8000, 5, 120, 90, 2],
    cov=np.diag([500**2, 1**2, 15**2, 5**2, 1**2]),
    size=80
)
c1 = np.random.multivariate_normal(
    mean=[1500, 20, 30, 40, 10],
    cov=np.diag([300**2, 4**2, 8**2, 10**2, 3**2]),
    size=100
)
c2 = np.random.multivariate_normal(
    mean=[3500, 10, 65, 60, 5],
    cov=np.diag([400**2, 2**2, 10**2, 8**2, 2**2]),
    size=120
)

X_retail = np.vstack([c0, c1, c2])
y_retail = np.array([0]*80 + [1]*100 + [2]*120)
segment_names = {0: 'Premium', 1: 'Bargain Hunter', 2: 'Mainstream'}

# Scale before PCA
scaler = StandardScaler()
X_retail_scaled = scaler.fit_transform(X_retail)

# Scree plot for the retail data
pca_retail_full = PCA(random_state=42)
pca_retail_full.fit(X_retail_scaled)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 6), pca_retail_full.explained_variance_ratio_ * 100,
       color='steelblue', alpha=0.8)
ax.plot(range(1, 6),
        np.cumsum(pca_retail_full.explained_variance_ratio_) * 100,
        'o--', color='crimson', label='Cumulative')
ax.axhline(y=95, color='darkorange', linestyle=':', label='95% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('Scree Plot: Retail Customer Dataset (5 features)')
ax.legend()
plt.tight_layout()
plt.show()

for i, r in enumerate(pca_retail_full.explained_variance_ratio_):
    print(f'PC{i+1}: {r*100:.1f}% variance')

In [ ]:
np.random.seed(42)

# Project retail data to 2D
pca_retail_2d = PCA(n_components=2, random_state=42)
X_retail_2d = pca_retail_2d.fit_transform(X_retail_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['crimson', 'steelblue', 'darkorange']

for label, color in zip([0, 1, 2], colors):
    mask = y_retail == label
    ax.scatter(
        X_retail_2d[mask, 0], X_retail_2d[mask, 1],
        c=color, s=30, alpha=0.6,
        label=segment_names[label]
    )

ax.set_xlabel(f'PC1 ({pca_retail_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_retail_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('Retail Customer Segments in 2D PCA Space')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

def plot_biplot(X_2d_proj, components, feature_names, labels=None,
                label_names=None, title='Biplot'):
    """Create a PCA biplot: scatter of projected samples + loading vectors.
    
    The arrows show how each original feature maps to the PC1/PC2 plane.
    Arrow direction = which PC the feature loads onto.
    Arrow length = strength of the loading (how much the feature contributes).
    """
    fig, ax = plt.subplots(figsize=(9, 7))
    
    # Scatter plot of projected samples
    if labels is not None:
        unique_labels = np.unique(labels)
        cmap = plt.get_cmap('tab10')
        for lab in unique_labels:
            mask = labels == lab
            name = label_names[lab] if label_names else str(lab)
            ax.scatter(X_2d_proj[mask, 0], X_2d_proj[mask, 1],
                       color=cmap(lab / len(unique_labels)),
                       s=20, alpha=0.5, label=name)
    else:
        ax.scatter(X_2d_proj[:, 0], X_2d_proj[:, 1],
                   color='steelblue', s=20, alpha=0.5)
    
    # Scale the loading vectors to fit within the scatter plot range
    x_scale = (X_2d_proj[:, 0].max() - X_2d_proj[:, 0].min()) * 0.4
    y_scale = (X_2d_proj[:, 1].max() - X_2d_proj[:, 1].min()) * 0.4
    
    # Draw one arrow per original feature
    for i, (feat_name, loading) in enumerate(zip(feature_names, components.T)):
        # loading[0] = projection of feature i onto PC1
        # loading[1] = projection of feature i onto PC2
        ax.annotate('',
                    xy=(loading[0] * x_scale, loading[1] * y_scale),
                    xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='crimson', lw=1.8))
        ax.text(loading[0] * x_scale * 1.15,
                loading[1] * y_scale * 1.15,
                feat_name,
                color='crimson', fontsize=10, fontweight='bold', ha='center')
    
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)
    if labels is not None:
        ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    plt.show()

plot_biplot(
    X_retail_2d,
    components=pca_retail_2d.components_.T,   # shape (n_features, n_components)
    feature_names=feature_names,
    labels=y_retail,
    label_names=segment_names,
    title='Biplot: Retail Customer PCA\n(arrows = feature loading vectors)'
)

print('How to read the biplot:')
print('  - Arrow pointing RIGHT  → high positive loading on PC1')
print('  - Arrow pointing UP     → high positive loading on PC2')
print('  - Long arrow            → feature has strong influence in PC1/PC2 plane')
print('  - Arrows in same direction → features are positively correlated')
print('  - Arrows opposite       → features are negatively correlated')
print()
print('Component loadings (rows = PCs, columns = features):')
loadings_df = pd.DataFrame(
    np.round(pca_retail_2d.components_, 3),
    columns=feature_names,
    index=['PC1', 'PC2']
)
print(loadings_df)

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You have a dataset with 100 features. After running PCA, you find that the first 10 components explain 95% of the variance. You train a logistic regression on these 10 components. A colleague says "you lost 5% of the information — the model will be 5% less accurate." Is this reasoning correct?

<details><summary>Show answer</summary>

No, this reasoning is flawed in two ways. First, the 5% discarded variance is often **noise**, not signal. The compressed model may actually be **more** accurate than one trained on all 100 features, because removing noise can reduce overfitting. Second, explained variance ratio describes variance in *X* (the features), not in *y* (the target). A direction that captures a lot of variance in *X* might have little predictive power for *y*, and vice versa. The correct approach is to compare model performance on a held-out test set with and without PCA, not to reason from the variance ratio alone.

</details>

---

**Question 2:** In the SVD factorisation $X_c = U \Sigma V^\top$, what are the principal component **scores** (the projected data) and what are the principal component **loadings** (the directions)?

<details><summary>Show answer</summary>

- **Loadings (directions):** the columns of $V$, equivalently the rows of $V^\top$. These are the directions in the original feature space along which variance is maximised. Shape: $(d, k)$ for the top-*k* PCs.
- **Scores (projected data):** the matrix $Z = X_c V_k = U_k \Sigma_k$ where $U_k$ and $\Sigma_k$ contain the first *k* columns/values. Each row of $Z$ is one data point expressed in the PC coordinate system. Shape: $(n, k)$.

In sklearn: `pca.components_` = loadings (rows are PCs), `pca.transform(X)` = scores.

</details>

---

**Question 3:** You apply PCA to your retail customer data (5 features → 2 PCs) and notice that PC1 has high positive loadings on `annual_spend` and `avg_basket` and high negative loading on `returns_rate`. How would you name and interpret this component for a business audience?

<details><summary>Show answer</summary>

PC1 can be interpreted as a **"Customer Value"** axis. Customers at the positive end of PC1 spend a lot (`annual_spend` ↑), buy in large amounts per visit (`avg_basket` ↑), and return very little (`returns_rate` ↓) — these are highly valuable, low-risk customers. Customers at the negative end spend less, buy small amounts, and return more items — lower-value, higher-friction customers. This is exactly the kind of actionable business interpretation that makes PCA useful for segmentation: each PC can often be named after the latent business concept it captures.

</details>

## Key Takeaways

1. **PCA finds the axes of maximum variance.** The first principal component is the direction in which your data spreads the most. The second is the perpendicular direction with the next-most spread.

2. **SVD is the workhorse.** Centre your data, compute $X_c = U \Sigma V^\top$, and the rows of $V^\top$ are your principal component directions. The explained variance ratio of PC *k* is $\sigma_k^2 / \sum \sigma_j^2$.

3. **The scree plot guides your choice of *k*.** Look for the elbow or use the 95% cumulative variance rule as a starting heuristic.

4. **Always scale your features first.** Without standardisation, features with large numerical ranges will dominate PCA.

5. **PCA is not magic for classification.** It maximises variance in *X*, not predictability of *y*. Evaluate downstream task performance empirically.

6. **Biplots connect PCA back to your original features.** They show which features drive which principal components — essential for business interpretation.

---

### What's Next

**t-SNE (t-Distributed Stochastic Neighbour Embedding)** addresses PCA's biggest limitation: it handles **non-linear** dimensionality reduction. Where PCA finds global linear structure, t-SNE finds local non-linear clusters. The next notebook will apply t-SNE to the same retail and digits data, and you will see dramatically better cluster separation — at the cost of interpretability and the ability to project new points.